# Local dry run — CPU stages on a Mac

Everything here also runs on a free Kaggle CPU session. The point of running it locally
first is that a Kaggle session costs a commit cycle to iterate on, while this costs
seconds.

**Deliberately avoids torchcodec.** `datasets` 4.x decodes audio through torchcodec,
which needs a matching torch build *and* a system FFmpeg. We cast to `Audio(decode=False)`
and decode with `soundfile` instead — which has a second benefit: that branch of
`audio_io.decode_audio_field` has never been exercised against this corpus's real audio
format. Kaggle proved the `audio_decoder` path; this proves the fallback.

**The token is never written to disk.** Cell 4 prompts for it with `getpass`, so it lives
in memory only. Do not paste it into a cell, and clear outputs before committing.

| Section | Needs network | Needs torch |
|---|---|---|
| 1–2 Setup | pip only | no |
| 3 Offline logic checks | **no** | no |
| 4–6 Corpus probe + decode | yes | no |
| 7 Mini profile | yes | no |
| 8 VAD segmentation | yes (torch.hub) | **yes** |
| 9 Shard round-trip | no | no |


## 1. Install

One-time. `torch` is optional and only needed for section 8.

In [ ]:
# Core: no torch, no torchcodec, no ffmpeg needed.
%pip install -q datasets soundfile numpy

# Optional, for the VAD section only. Large download (~100 MB+); skip if you just want
# the data-path checks.
# %pip install -q torch --index-url https://download.pytorch.org/whl/cpu


## 2. Point at the repo

Set `REPO` to wherever you cloned it. `sys.path.insert` rather than `pip install -e`:
we want the working tree, so an edit is picked up by a kernel restart and nothing else.

In [ ]:
import sys, platform
from pathlib import Path

REPO = Path.home() / "Desktop/Personal/Whisper_Distill"   # <-- edit if yours differs
WORK = REPO / "local_out"                                  # gitignored scratch
WORK.mkdir(exist_ok=True)

assert (REPO / "src/whisper_distill").is_dir(), f"no package under {REPO}"
sys.path.insert(0, str(REPO / "src"))

print(f"python {platform.python_version()} on {platform.machine()}")
print(f"repo   {REPO}")
print(f"work   {WORK}")

# This package is the one an unanchored `data/` gitignore rule silently excluded once.
# If this import fails, the clone is older than commit aa0a098.
import whisper_distill.data.audio_io  # noqa: F401
print("data package importable")


## 3. Offline logic checks

No network, no torch. Catches import errors and logic regressions in about a second, so
run this after every edit before touching the network sections.

In [ ]:
import numpy as np

from whisper_distill.config import DEFAULT
from whisper_distill.itn import normalise
from whisper_distill.evaluation.metrics import wer, report, code_mix_bucket
from whisper_distill.modeling.surgery import maximally_spaced_indices, select_vocabulary
from whisper_distill.labeling.quota import decode_quota_probe, implied_runtimes
from whisper_distill.data.segment import Segment, merge_to_window

checks = []
def check(name, got, want):
    ok = got == want
    checks.append(ok)
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
    if not ok:
        print(f"        want {want!r}\n        got  {got!r}")

a = DEFAULT.audio
print("feature geometry")
check("10s -> 1000 mel frames", a.n_frames, 1000)
check("1000 frames -> 500 encoder positions", a.n_encoder_positions, 500)
check("cache GB for 72k clips", round(a.cache_bytes(72_000) / 1e9, 2), 11.52)

print("\ninverse text normalisation")
check("code-switch passthrough",
      normalise("meeting chaar baje hai, Slack pe ping karo"),
      "meeting 4 baje hai, Slack pe ping karo")
check("half past",   normalise("saade chaar baje"), "4:30 baje")
check("quarter to",  normalise("paune paanch baje"), "4:45 baje")
check("composition", normalise("do hazaar bees"), "2020")
check("currency",    normalise("teen sau paanch rupaye"), "\u20b9305")

print("\nscoring")
check("wer counts substitution", wer("a b c", "a x c"), (1, 3))
check("punctuation only in punctuated variant",
      (wer("hi, there", "hi there")[0], wer("hi, there", "hi there", punctuated=True)[0]),
      (0, 1))
check("report pools rather than averages",
      round(report(["a", "b c d e f g h i j k"], ["a", "x x x x x x x x x x"]).overall["wer"], 4),
      round(10 / 11, 4))

print("\nmodel surgery")
check("12 -> 4 maximally spaced", maximally_spaced_indices(12, 4), [0, 4, 7, 11])
check("2 layers = first and last", maximally_spaced_indices(32, 2), [0, 31])
check("mandatory tokens survive",
      set([0, 1, 2]) <= set(select_vocabulary({i: 999 for i in range(100, 200)}, 10, [0, 1, 2])),
      True)

print("\nsegmentation")
segs = merge_to_window([Segment(0, 4), Segment(4.2, 8), Segment(8.1, 14)],
                       max_seconds=10, min_seconds=1)
check("window ceiling respected", all(s.duration_s <= 10.001 for s in segs), True)
check("25s region splits into 10/10/5",
      [round(s.duration_s, 1) for s in merge_to_window([Segment(0, 25)], max_seconds=10, min_seconds=1)],
      [10.0, 10.0, 5.0])

print("\nquota decoding")
check("two 15-min sessions, +30 -> wall clock",
      decode_quota_probe(meter_delta_minutes=30, session_minutes=[15, 15]).billing,
      "wall_clock")
check("midpoint refused",
      decode_quota_probe(meter_delta_minutes=45, session_minutes=[15, 15]).billing,
      "ambiguous")
check("observed 54 min inverse solve",
      (implied_runtimes(54, n_sessions=2).if_wall_clock, implied_runtimes(54, n_sessions=2).if_per_gpu),
      (54.0, 27.0))

print(f"\n{sum(checks)}/{len(checks)} offline checks passed")
assert all(checks), "fix these before running anything that costs a session"


## 4. Hugging Face token

`getpass`, so it stays in memory. On Kaggle this cell becomes
`UserSecretsClient().get_secret("HF_TOKEN")` — that is the only difference.

Read scope is enough. Accept the dataset terms on the HF website first, or this 403s.

In [ ]:
from getpass import getpass

HF_TOKEN = getpass("HF token (read scope, hidden): ").strip()
print(f"token loaded: {len(HF_TOKEN)} chars, starts {HF_TOKEN[:3]!r}")
assert HF_TOKEN.startswith("hf_"), "that does not look like an HF token"


## 5. Schema probe — Vaani

Ten seconds. Confirms gating, columns, and which decode branch runs. `describe_row`
discovers the audio and text columns instead of assuming their names.

In [ ]:
from datasets import Audio, load_dataset
from whisper_distill.data.audio_io import describe_row, find_audio_key, find_transcript_key
from whisper_distill.data.streaming import take

VAANI = ("ARTPARK-IISc/Vaani-transcription-part", "Hindi", "train")

ds = load_dataset(VAANI[0], VAANI[1], split=VAANI[2], streaming=True, token=HF_TOKEN)
# decode=False keeps torchcodec (and system FFmpeg) out of the picture entirely.
ds = ds.cast_column("audio", Audio(decode=False))

row = next(iter(take(ds, 1)))
print(describe_row(row))


## 6. Decode check — the soundfile branch

This is the branch Kaggle did **not** take, so nothing has yet proven it works on
Vaani's actual audio format. If it decodes here, it is a validated fallback for Kaggle
too; if it fails, we learn the container format and fix `audio_io` before it matters.

In [ ]:
from whisper_distill.data.audio_io import decode_audio_field

audio_key = find_audio_key(row)
text_key = find_transcript_key(row)
print(f"audio={audio_key!r}  text={text_key!r}")

raw = row[audio_key]
if isinstance(raw, dict):
    print(f"path: {str(raw.get('path'))[-60:]}")
    print(f"bytes: {len(raw['bytes']) if raw.get('bytes') else 0}")

wav, how = decode_audio_field(raw)
print(f"\ndecoded via {how!r}")
print(f"  {len(wav) / 16000:.2f} s at 16 kHz, dtype {wav.dtype}, shape {wav.shape}")
print(f"  peak {abs(wav).max():.4f}, rms {(wav ** 2).mean() ** 0.5:.4f}")
assert wav.ndim == 1 and wav.dtype == np.float32, "expected mono float32"
assert abs(wav).max() > 1e-4, "decoded to near-silence -- suspect the decode path"
print("\nsoundfile branch works on this corpus")


## 7. Mini profile

40 rows rather than 400 — enough to catch a crash, not enough for stable deciles. Kaggle
runs the real 400-row profile via `kaggle/01a_corpus_distribution_cpu.py`.

Change `SOURCE` to point at IndicVoices once its config and split are known.

In [ ]:
import re, time
from collections import Counter
from whisper_distill.evaluation.metrics import code_mix_density

SOURCE = VAANI          # or ("ai4bharat/IndicVoices", "hindi", "train")
N_ROWS = 40

def profile(source, n_rows):
    name, config, split = source
    d = load_dataset(name, config, split=split, streaming=True, token=HF_TOKEN)
    d = d.cast_column("audio", Audio(decode=False))

    first = next(iter(take(d, 1)))
    a_key, t_key = find_audio_key(first), find_transcript_key(first)
    if t_key is None:
        raise SystemExit(f"no text column among {sorted(first)}")
    print(f"{name}:{config}:{split}  audio={a_key!r} text={t_key!r}\n")
    del first

    d = load_dataset(name, config, split=split, streaming=True, token=HF_TOKEN)
    d = d.cast_column("audio", Audio(decode=False))
    latin = re.compile(r"[A-Za-z]")
    durations, densities, buckets = [], [], Counter()
    n_latin = n_comma = n_empty = 0
    t0 = time.time()

    for i, r in enumerate(take(d, n_rows)):
        try:
            w, _ = decode_audio_field(r[a_key])
        except Exception as e:
            print(f"  row {i}: {type(e).__name__}; skipped")
            continue
        durations.append(len(w) / 16000)
        t = (r.get(t_key) or "").strip()
        if not t:
            n_empty += 1
            continue
        densities.append(code_mix_density(t))
        buckets[code_mix_bucket(t)] += 1
        n_latin += bool(latin.search(t))
        n_comma += ("," in t or "\u060c" in t)

    q = lambda v, p: sorted(v)[min(len(v) - 1, int(round(p * (len(v) - 1))))] if v else 0.0
    print(f"{len(durations)} rows in {time.time() - t0:.1f} s\n")
    print("duration s : " + "  ".join(
        f"{lbl}={q(durations, p):.2f}" for lbl, p in
        (("p10", .1), ("p50", .5), ("p90", .9), ("max", 1.0))))
    print(f"over 10 s  : {sum(d > 10 for d in durations)}/{len(durations)}")
    print(f"empty text : {n_empty}")
    print(f"latin      : {n_latin}/{len(densities)} "
          f"({n_latin / max(len(densities), 1) * 100:.0f}%)")
    print(f"commas     : {n_comma}/{len(densities)}")
    print(f"mix density: {sum(densities) / max(len(densities), 1):.3f}")
    print(f"buckets    : {dict(buckets)}")
    return durations

durations = profile(SOURCE, N_ROWS)


## 8. VAD segmentation — needs torch

Skips itself cleanly if torch is not installed. Uninstructive on Vaani anyway: measured
median clip is 2.60 s, so there is little to segment. This exists to prove the code path
runs, and it will matter for the scraped long-form audio.

In [ ]:
try:
    import torch
except ImportError:
    print("torch not installed -- skipping. Uncomment the torch line in cell 1 to run it.")
else:
    model, utils = torch.hub.load("snakers4/silero-vad", "silero_vad", trust_repo=True)
    get_speech_timestamps = utils[0]

    stamps = get_speech_timestamps(
        torch.from_numpy(np.ascontiguousarray(wav)), model,
        sampling_rate=16000, return_seconds=True,
    )
    speech = [Segment(float(s["start"]), float(s["end"])) for s in stamps]
    clips = merge_to_window(speech, max_seconds=10, min_seconds=1)

    print(f"clip is {len(wav) / 16000:.2f} s")
    print(f"raw VAD regions : {[(round(s.start_s, 2), round(s.end_s, 2)) for s in speech]}")
    print(f"after merge     : {[(round(s.start_s, 2), round(s.end_s, 2)) for s in clips]}")
    speech_s = sum(s.duration_s for s in speech)
    print(f"speech {speech_s:.2f} s of {len(wav) / 16000:.2f} s "
          f"({speech_s / (len(wav) / 16000) * 100:.0f}%)")


## 9. Shard round-trip

No teacher locally, so the mels are synthetic — but the writer, the padding convention,
the index and the memory-mapped read are the real code.

The padding assertion is the one that matters. Whisper normalises log-mel as
`(log10(mag) + 4) / 4` after flooring at `log_spec.max() - 8`, so its padded frames hold a
constant exactly `2.0` below the clip maximum — **not zero**. With a frozen encoder there
is no adapting to a different convention, so zero-padding would be a silent
train/inference mismatch across the whole corpus.

In [ ]:
from whisper_distill.data.pack import (
    WHISPER_FLOOR_OFFSET, PAD_TOKEN, ShardWriter, open_shards, whisper_floor,
)

cache = WORK / "shard_test"
import shutil; shutil.rmtree(cache, ignore_errors=True)

with ShardWriter(cache, n_mels=80, n_frames=1000, max_tokens=32,
                 shard_target_bytes=600_000) as w:
    # short mel -> writer must pad with the Whisper floor
    w.add("short", np.full((80, 260), 0.75, dtype=np.float32), [1, 2, 3],
          source="synthetic", duration_s=2.6, teacher_wer=0.05)
    # full-width mel with its own floor tail -> stored untouched (the preferred path)
    full = np.full((80, 1000), 0.40, dtype=np.float32); full[:, 600:] = -0.83
    w.add("full", full, [4, 5], speech_frames=600,
          source="synthetic", duration_s=6.0, teacher_wer=0.30)
    for i in range(6):
        w.add(f"pad{i}", np.full((80, 300), 0.5, dtype=np.float32), [9],
              source="synthetic", duration_s=3.0, teacher_wer=0.10)

index, shards = open_shards(cache)
print(f"{len(index.records)} clips across {len(shards)} shards")
print(f"files: {sorted(p.name for p in cache.iterdir())}\n")

r = index.records[0]
mels, toks = shards[r.shard]
tail = np.asarray(mels[r.row][:, 260:])
print(f"short clip: speech frames {r.n_frames}, padded value {tail.flat[0]:.4f}")
print(f"  expected whisper floor  {0.75 - WHISPER_FLOOR_OFFSET:.4f}")
assert not np.any(tail == 0.0), "padding must never be zero"
assert np.allclose(tail, 0.75 - WHISPER_FLOOR_OFFSET, atol=1e-3)
assert toks[r.row][3] == PAD_TOKEN
print("  PASS  padded with the whisper floor, not zeros")

kept = index.filtered(max_teacher_wer=0.20)
print(f"\nWER<=0.20 keeps {len(kept)}/{len(index.records)} "
      f"(drops the 0.30 clip)")
assert len(kept) == len(index.records) - 1
print("  PASS  filter applied on read")
shutil.rmtree(cache, ignore_errors=True)


## What changes on Kaggle

| Local | Kaggle |
|---|---|
| `getpass("HF token")` | `UserSecretsClient().get_secret("HF_TOKEN")` |
| `REPO = ~/Desktop/...` | `/kaggle/working/whisper_distill` (or `/kaggle/tmp/...`) |
| `WORK = REPO/local_out` | `/kaggle/working/...` |
| `cast_column(Audio(decode=False))` | omit — torchcodec works there, and the `audio_decoder` path is proven on that image |
| `N_ROWS = 40` | 400, via `kaggle/01a_corpus_distribution_cpu.py` |
| `%pip install datasets soundfile` | preinstalled; only `soundfile` is worth pinning |

Nothing else. Sections 3 and 9 need no network at all, so run them after any edit —
they are the cheapest regression check in the project.

**Before committing this notebook: Kernel → Restart & Clear Output.** Cell 4 output is
harmless (it prints a length, not the token) but clearing is the habit worth having.